In [ ]:
# 1 安装基础的依赖包
sudo yum install -y bcc bpftrace ipmitool smartmontools lm_sensors \
                    stress-ng chaosblade docker-ce prometheus2 grafana \
                    python3.10 python3.10-pip git

In [ ]:
# 2. 全栈观测体系搭建
搭建H/K/R/A 四层统一观测平台，采集 BCPN 模型所需的所有指标，采样间隔严格设置为 10 秒（与论文一致）。
## 2.1 部署 Prometheus+Grafana
# 启动Prometheus
sudo systemctl enable --now prometheus
sudo systemctl enable --now grafana-server

# 配置Prometheus采集目标（/etc/prometheus/prometheus.yml）
sudo tee /etc/prometheus/prometheus.yml <<EOF
global:
  scrape_interval: 10s  # 严格对齐论文采样间隔
  evaluation_interval: 10s

scrape_configs:
  - job_name: 'node'  # 硬件层+内核层指标
    static_configs:
      - targets: ['localhost:9100']
  - job_name: 'cadvisor'  # 运行时层（容器）指标
    static_configs:
      - targets: ['localhost:8080']
  - job_name: 'app'  # 应用层指标
    static_configs:
      - targets: ['localhost:8000']
EOF

sudo systemctl restart prometheus

# 检查服务状态
systemctl status prometheus --no-pager
# 预期：active (running)

# 检查端口监听
ss -tulpn | grep 9090
# 预期：LISTEN 0.0.0.0:9090

# 访问API验证
curl -s http://localhost:9090/api/v1/status/config | head -20
# 预期：返回JSON格式的配置信息

In [ ]:
## 2.2 部署各层指标采集器
### （1）硬件层 + 内核层：Node Exporter,采集指标：CPU 使用率、内存使用率、磁盘 IO、磁盘温度、网络流量、内核调度延迟等。
docker run -d --name node-exporter --net=host --pid=host \
  -v "/:/host:ro,rslave" \
  quay.io/prometheus/node-exporter:latest \
  --path.rootfs=/host


# 检查容器状态
docker ps | grep node-exporter
# 预期：运行中，端口9100

# 检查指标输出
curl -s http://localhost:9100/metrics | grep node_disk_io_time_seconds_total
# 预期：返回类似 node_disk_io_time_seconds_total{device="sda"} 12345.67 的数值

In [ ]:
### （2）运行时层：cAdvisor, 采集指标：容器 CPU / 内存 / IO 使用率、容器重启次数、OOM 事件等。
docker run -d --name cadvisor --net=host \
  -v /:/rootfs:ro \
  -v /var/run:/var/run:ro \
  -v /sys:/sys:ro \
  -v /var/lib/docker/:/var/lib/docker:ro \
  gcr.io/cadvisor/cadvisor:latest

# 查看容器状态
docker ps | grep cadvisor

# 访问 cAdvisor Web 界面（默认端口 8080）
curl http://localhost:8080/metrics

In [ ]:
### （3）应用层：自定义 Exporter
编写一个简单的 Python 应用，模拟电商订单服务并暴露指标：
# app_exporter.py
from prometheus_client import start_http_server, Counter, Gauge
import time
import random

# 应用层指标
REQUESTS = Counter('app_requests_total', 'Total requests')
RETRIES = Counter('app_retries_total', 'Total retries')
RESPONSE_TIME = Gauge('app_response_time_ms', 'Average response time')

def mock_order_service():
    """模拟订单服务，故障时会触发重试风暴"""
    REQUESTS.inc()
    # 正常响应时间100ms，故障时500ms以上
    if time.time() > fault_start_time:
        RESPONSE_TIME.set(random.normalvariate(600, 100))
        # 30%概率触发重试
        if random.random() < 0.3:
            RETRIES.inc()
            time.sleep(0.1)
    else:
        RESPONSE_TIME.set(random.normalvariate(100, 20))

if __name__ == '__main__':
    start_http_server(8000)
    fault_start_time = time.time() + 300  # 5分钟后注入故障
    print("应用启动成功，指标端口：8000")
    while True:
        mock_order_service()
        time.sleep(0.1)

In [ ]:
# 3 故障 1：硬件层 SSD 介质坏道（论文核心案例）
## 3.1 对应论文 6.1 节真实生产案例，根因为 SSD 硬件故障，叠加应用重试形成反馈环。
# 1. 先启动正常业务负载
python app_exporter.py &
docker run -d --name mysql -e MYSQL_ROOT_PASSWORD=123456 -p 3306:3306 mysql:8.0

# 检查app_exporter
curl -s http://localhost:8000/metrics | grep app_requests_total
    


# 2. 5分钟后注入SSD故障（模拟介质坏道导致IO延迟飙升）
# 方法1：用dd模拟磁盘IO饱和（安全）
sudo dd if=/dev/sda of=/dev/null bs=1M count=100000 &

# 方法2：用chaosblade注入磁盘IO错误（更真实）
curl -L https://chaosblade.oss-cn-hangzhou.aliyuncs.com/agent/github/1.7.2/chaosblade-1.7.2-linux-amd64.tar.gz | tar xz
cd chaosblade-1.7.2-linux-amd64
./blade create disk fill --path=/ --size=1024 --timeout=300s

In [ ]:
## 3.2. 故障 2：内核层调度延迟
模拟内核调度器负载过高导致的进程调度延迟：
# 启动100个CPU密集型进程，耗尽CPU时间片
stress-ng --cpu 100 --timeout 300s

In [ ]:
### 3.3. 故障 3：运行时层容器 CPU 饱和
模拟容器 CPU 配额不足导致的服务卡顿：
# 启动一个CPU受限的容器，然后压满
docker run -d --name cpu-limit --cpus=0.5 polinux/stress stress --cpu 1 --timeout 300s

In [ ]:
### 3.4. 故障 4：应用层重试风暴（反馈环核心场景）
对应论文 3.4 节反馈环建模，形成应用重试→容器CPU高→内核调度慢→应用超时→更多重试的自激环
# 修改app_exporter.py中的重试概率为0.8，触发重试风暴
sed -i 's/random.random() < 0.3/random.random() < 0.8/g' app_exporter.py
pkill -f app_exporter.py
python app_exporter.py &

In [ ]:
# 4 自动化数据采集与 BCPN 格式转换
## 编写脚本自动从 Prometheus 拉取故障时间窗口内的指标，生成 BCPN 模型所需的多 CSV 格式数据，完全兼容之前的代码。
## 1. 数据采集脚本（generate_bcpn_data.py）
from prometheus_api_client import PrometheusConnect
import pandas as pd
import json
from datetime import datetime, timedelta

# 配置Prometheus地址
PROM_URL = "http://localhost:9090"
# 故障时间窗口（根据实际故障注入时间调整）
START_TIME = datetime.now() - timedelta(minutes=10)
END_TIME = datetime.now()
STEP = "10s"  # 与采样间隔一致

# 定义BCPN四层节点与对应Prometheus指标
BCPN_NODES = [
    {
        "node_id": "host01_ssd_io_util",
        "layer": "H",
        "metric": "node_disk_io_time_seconds_total{device=\"sda\"}",
        "has_inherent_defect": True,
        "node_description": "SSD磁盘IO利用率"
    },
    {
        "node_id": "kernel_sched_latency",
        "layer": "K",
        "metric": "node_schedstat_waiting_seconds_total",
        "has_inherent_defect": False,
        "node_description": "内核调度等待延迟"
    },
    {
        "node_id": "docker_mysql_cpu",
        "layer": "R",
        "metric": "container_cpu_usage_seconds_total{name=\"mysql\"}",
        "has_inherent_defect": False,
        "node_description": "MySQL容器CPU使用率"
    },
    {
        "node_id": "order_service_retries",
        "layer": "A",
        "metric": "app_retries_total",
        "has_inherent_defect": False,
        "node_description": "订单服务重试次数"
    }
]

def main():
    prom = PrometheusConnect(url=PROM_URL, disable_ssl=True)
    os.makedirs("bcpn_test_scenario/nodes", exist_ok=True)
    
    # 1. 生成每个节点的CSV文件
    for node in BCPN_NODES:
        print(f"采集节点：{node['node_id']}")
        data = prom.get_metric_range_data(
            metric_name=node["metric"],
            start_time=START_TIME,
            end_time=END_TIME,
            step=STEP
        )
        
        if not data:
            print(f"警告：未采集到节点 {node['node_id']} 的数据")
            continue
            
        # 转换为DataFrame
        df = pd.DataFrame(data[0]["values"], columns=["time", "value"])
        df["time"] = pd.to_datetime(df["time"], unit="s")
        df.to_csv(f"bcpn_test_scenario/nodes/{node['node_id']}.csv", index=False)
    
    # 2. 生成场景元数据文件
    scenario_metadata = {
        "scenario_info": {
            "scenario_id": "ssd_fault_with_retry_loop",
            "scenario_name": "SSD介质坏道叠加应用重试风暴",
            "fault_start_time": START_TIME.isoformat(),
            "fault_end_time": END_TIME.isoformat(),
            "affected_hosts": ["host01"]
        },
        "nodes": BCPN_NODES,
        "logs": []
    }
    
    with open("bcpn_test_scenario/scenario_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scenario_metadata, f, ensure_ascii=False, indent=2)
    
    print("✅ BCPN格式数据生成完成，路径：./bcpn_test_scenario/")

if __name__ == "__main__":
    main()

In [ ]:
# 2. 运行数据采集
# 故障注入完成后，运行采集脚本
python3.10 generate_bcpn_data.py